In [1]:
!pip install -q -U keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 3.8 MB/s eta 0:00:00


In [2]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
from sklearn.model_selection import KFold
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import keras_tuner as kt  # Keras Tuner

Mounted at /content/drive


In [3]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

In [4]:
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your actual path to the dataset


In [5]:
from keras.saving import register_keras_serializable

@register_keras_serializable()
class PAM(tf.keras.layers.Layer):
    """Position Attention Module"""
    def __init__(self):
        super(PAM, self).__init__()

    def build(self, input_shape):
        self.query_conv = tf.keras.layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.key_conv = tf.keras.layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.value_conv = tf.keras.layers.Conv2D(input_shape[-1], kernel_size=1)
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        query = self.query_conv(inputs)  # [batch, h, w, c//8]
        key = self.key_conv(inputs)  # [batch, h, w, c//8]
        value = self.value_conv(inputs)  # [batch, h, w, c]

        # Compute attention
        query = tf.reshape(query, [tf.shape(query)[0], -1, tf.shape(query)[-1]])  # [batch, hw, c//8]
        key = tf.transpose(tf.reshape(key, [tf.shape(key)[0], -1, tf.shape(key)[-1]]), perm=[0, 2, 1])  # [batch, c//8, hw]
        energy = tf.matmul(query, key)  # [batch, hw, hw]
        attention = tf.nn.softmax(energy, axis=-1)  # Spatial attention

        value = tf.reshape(value, [tf.shape(value)[0], -1, tf.shape(value)[-1]])  # [batch, hw, c]
        out = tf.matmul(attention, value)  # [batch, hw, c]
        out = tf.reshape(out, tf.shape(inputs))  # Restore shape
        return self.gamma * out + inputs


@register_keras_serializable()
class CAM(tf.keras.layers.Layer):
    """Channel Attention Module"""
    def __init__(self):
        super(CAM, self).__init__()

    def build(self, input_shape):
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        # Compute attention
        query = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])  # [batch, hw, c]
        key = tf.transpose(query, perm=[0, 2, 1])  # [batch, c, hw]
        energy = tf.matmul(key, query)  # [batch, c, c]
        attention = tf.nn.softmax(energy, axis=-1)  # Channel attention

        value = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])  # [batch, hw, c]
        out = tf.matmul(value, attention)  # [batch, hw, c]
        out = tf.reshape(out, tf.shape(inputs))  # Restore shape
        return self.gamma * out + inputs




In [6]:
@register_keras_serializable()
class DANetBlock(tf.keras.layers.Layer):
    """Dual Attention Network Block"""
    def __init__(self):
        super(DANetBlock, self).__init__()
        self.pam = PAM()
        self.cam = CAM()

    def call(self, inputs):
        pam_out = self.pam(inputs)
        cam_out = self.cam(inputs)
        return pam_out + cam_out  # Combine PAM and CAM outputs


In [7]:
def build_model():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    # Convolutional Block 1
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 2
    x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 3
    x = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # DANet Block
    x = DANetBlock()(x)

    # Global Average Pooling and Fully Connected Layers
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    # Output Layer
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    # Model Compilation
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    learning_rate = 0.0004
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [8]:
# Test DANetBlock
x_test = tf.random.normal((1, 64, 64, 256))  # Example input
danet_block = DANetBlock()
x_out = danet_block(x_test)

print(f"Output shape: {x_out.shape}")


Output shape: (1, 64, 64, 256)


In [9]:

# Step 3: Load and prepare datasets with a given batch size
img_size = (256, 256)


def prepare_datasets(batch_size):
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Data Augmentation and Normalization
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
        tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

    return train_dataset, val_dataset

In [10]:

# Train and evaluate the model
batch_size = 16
train_dataset, val_dataset = prepare_datasets(batch_size=batch_size)

Found 7670 files belonging to 2 classes.
Using 6136 files for training.
Found 7670 files belonging to 2 classes.
Using 1534 files for validation.


In [11]:
model = build_model()
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30)



Epoch 1/30
147/384 ━━━━━━━━━━━━━━━━━━━━ 27:02 7s/step - accuracy: 0.7229 - loss: 4.6124

KeyboardInterrupt: 

In [ ]:
# Plot Loss and Accuracy During Training
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate the model on validation dataset
y_true = []
y_pred_probs = []

for batch in val_dataset.as_numpy_iterator():
    X, y = batch
    preds = model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(preds)

In [ ]:
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

# Plot PR Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_probs)
pr_auc = np.trapz(precision_vals, recall_vals)
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid()
plt.show()

In [ ]:
import os

In [ ]:
# Save the trained model
model_path = '/content/drive/My Drive/saved_models/danet_model.keras'  # Changed path
model.save(model_path)
print(f"DANet model saved at: {model_path}")

In [ ]:
global danet_y_true, danet_y_pred_probs
danet_y_true = y_true
danet_y_pred_probs = y_pred_probs

***PANET***

In [ ]:
!pip install -q -U keras-tuner


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
from sklearn.model_selection import KFold
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import keras_tuner as kt  # Keras Tuner

In [ ]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your actual path to the dataset


In [ ]:
from keras.saving import register_keras_serializable

@register_keras_serializable()
class PANetBlock(tf.keras.layers.Layer):
    def __init__(self, filters):
        super(PANetBlock, self).__init__()
        self.filters = filters

        # Convolution layers for channel adjustment
        self.conv_adjust_x1 = tf.keras.layers.Conv2D(filters, (1, 1), activation='relu', padding='same')  # Adjust channels for x1
        self.conv_adjust_x2 = tf.keras.layers.Conv2D(filters, (1, 1), activation='relu', padding='same')  # Adjust channels for x2

        # Layers for top-down and bottom-up fusion
        self.conv1 = tf.keras.layers.Conv2D(filters, (1, 1), activation='relu', padding='same')  # Top-down fusion
        self.conv2 = tf.keras.layers.Conv2D(filters, (3, 3), activation='relu', padding='same')  # Bottom-up fusion
        self.upsample = tf.keras.layers.UpSampling2D(size=(2, 2))
        self.downsample = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))

    def call(self, inputs):
        x1, x2 = inputs  # Inputs: Higher-level features (x1), Lower-level features (x2)

        # Adjust x1 and x2 channels to match filters
        x1_adjusted = self.conv_adjust_x1(x1)
        x2_adjusted = self.conv_adjust_x2(x2)

        # Top-down fusion
        x1_upsampled = self.upsample(x1_adjusted)
        x1_fused = tf.keras.layers.Add()([x1_upsampled, x2_adjusted])
        x1_fused = self.conv1(x1_fused)

        # Bottom-up fusion
        x2_downsampled = self.downsample(x2_adjusted)
        x2_fused = tf.keras.layers.Add()([x2_downsampled, x1_adjusted])
        x2_fused = self.conv2(x2_fused)

        return x1_fused, x2_fused



In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    # Convolutional Block 1
    x1 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x1 = tf.keras.layers.BatchNormalization()(x1)
    x1 = tf.keras.layers.MaxPooling2D((2, 2))(x1)

    # Convolutional Block 2
    x2 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x1)
    x2 = tf.keras.layers.BatchNormalization()(x2)
    x2 = tf.keras.layers.MaxPooling2D((2, 2))(x2)

    # Convolutional Block 3
    x3 = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x2)
    x3 = tf.keras.layers.BatchNormalization()(x3)
    x3 = tf.keras.layers.MaxPooling2D((2, 2))(x3)

    # PANet: Feature Aggregation
    panet_block = PANetBlock(filters=128)
    x1_panet, x2_panet = panet_block((x3, x2))

    # Global Average Pooling and Fully Connected Layers
    x = tf.keras.layers.GlobalAveragePooling2D()(x1_panet)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    # Output Layer
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    # Model Compilation
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    learning_rate = 0.0004
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:

# Step 3: Load and prepare datasets with a given batch size
img_size = (256, 256)


def prepare_datasets(batch_size):
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Data Augmentation and Normalization
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
        tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

    return train_dataset, val_dataset

In [ ]:

# Train and evaluate the model
batch_size = 16
train_dataset, val_dataset = prepare_datasets(batch_size=batch_size)

In [ ]:
# Create test inputs
x1_test = tf.random.normal((1, 32, 32, 128))  # Higher-level features
x2_test = tf.random.normal((1, 64, 64, 64))   # Lower-level features

# Initialize and test PANetBlock
panet_block = PANetBlock(filters=64)
x1_out, x2_out = panet_block((x1_test, x2_test))

print(f"x1_out shape: {x1_out.shape}")
print(f"x2_out shape: {x2_out.shape}")


In [ ]:
model = build_model()
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30)



In [ ]:
# Plot Loss and Accuracy During Training
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate the model on validation dataset
y_true = []
y_pred_probs = []

for batch in val_dataset.as_numpy_iterator():
    X, y = batch
    preds = model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(preds)

In [ ]:
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

# Plot PR Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_probs)
pr_auc = np.trapz(precision_vals, recall_vals)
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid()
plt.show()

In [ ]:
import os

In [ ]:
# Save the trained model
model_path = '/content/drive/My Drive/saved_models/panet_model.keras'  # Changed path
model.save(model_path)
print(f"PANet model saved at: {model_path}")

In [ ]:
# After evaluation, store metrics
global panet_y_true, panet_y_pred_probs
panet_y_true = y_true
panet_y_pred_probs = y_pred_probs

***CBAM***

In [ ]:
!pip install -q -U keras-tuner


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
from sklearn.model_selection import KFold
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import keras_tuner as kt  # Keras Tuner

In [ ]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your actual path to the dataset


In [ ]:
# Define CBAM module
class CBAM(tf.keras.layers.Layer):
    def __init__(self, filters, reduction_ratio=8):
        super(CBAM, self).__init__()
        self.filters = filters
        self.reduction_ratio = reduction_ratio

        # Channel attention layers
        self.global_avg_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.global_max_pool = tf.keras.layers.GlobalMaxPooling2D()
        self.dense1 = tf.keras.layers.Dense(filters // reduction_ratio, activation='relu', use_bias=False)
        self.dense2 = tf.keras.layers.Dense(filters, activation='sigmoid', use_bias=False)

        # Spatial attention layers
        self.conv2d = tf.keras.layers.Conv2D(1, kernel_size=7, activation='sigmoid', padding='same')

    def call(self, inputs):
        # Channel attention
        avg_pool = self.global_avg_pool(inputs)
        max_pool = self.global_max_pool(inputs)
        dense_out = self.dense2(self.dense1(avg_pool)) + self.dense2(self.dense1(max_pool))
        channel_attention = tf.expand_dims(tf.expand_dims(dense_out, axis=1), axis=1)
        x = inputs * channel_attention

        # Spatial attention
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        spatial_attention = self.conv2d(concat)

        return x * spatial_attention

In [ ]:
# Define the CNN model-building function with CBAM
def build_model():
    model = tf.keras.Sequential()

    # Convolutional Block 1
    model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
    model.add(CBAM(32))  # Add CBAM
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))

    # Convolutional Block 2
    model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(CBAM(64))  # Add CBAM
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))

    # Convolutional Block 3
    model.add(tf.keras.layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(CBAM(128))  # Add CBAM
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))

    # Global Average Pooling
    model.add(tf.keras.layers.GlobalAveragePooling2D())

    # Fully Connected Layers with Dropout
    model.add(tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)))
    model.add(tf.keras.layers.Dropout(0.4))
    model.add(tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)))
    model.add(tf.keras.layers.Dropout(0.4))

    # Output Layer
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    # Compile Model
    learning_rate = 0.0004  # Set best learning rate
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:

# Step 3: Load and prepare datasets with a given batch size
img_size = (256, 256)


def prepare_datasets(batch_size):
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Data Augmentation and Normalization
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
        tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

    return train_dataset, val_dataset

In [ ]:

# Train and evaluate the model
batch_size = 16
train_dataset, val_dataset = prepare_datasets(batch_size=batch_size)

In [ ]:
model = build_model()
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30)



In [ ]:
# Plot Loss and Accuracy During Training
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate the model on validation dataset
y_true = []
y_pred_probs = []

for batch in val_dataset.as_numpy_iterator():
    X, y = batch
    preds = model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(preds)

In [ ]:
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

# Plot PR Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_probs)
pr_auc = np.trapz(precision_vals, recall_vals)
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid()
plt.show()

In [ ]:
import os

In [ ]:
model_path = '/content/drive/My Drive/saved_models/cbam_model.keras'  # Changed path
model.save(model_path)
print(f"CBAM model saved at: {model_path}")

In [ ]:
global cbam_y_true, cbam_y_pred_probs
cbam_y_true = y_true
cbam_y_pred_probs = y_pred_probs


In [ ]:
# Combined plotting code (added to last model)
# ROC Curve
plt.figure(figsize=(10, 6))
fpr_danet, tpr_danet, _ = roc_curve(danet_y_true, danet_y_pred_probs)
roc_auc_danet = roc_auc_score(danet_y_true, danet_y_pred_probs)
plt.plot(fpr_danet, tpr_danet, label=f'DANet (AUC={roc_auc_danet:.2f})')

fpr_panet, tpr_panet, _ = roc_curve(panet_y_true, panet_y_pred_probs)
roc_auc_panet = roc_auc_score(panet_y_true, panet_y_pred_probs)
plt.plot(fpr_panet, tpr_panet, label=f'PANet (AUC={roc_auc_panet:.2f})')

fpr_cbam, tpr_cbam, _ = roc_curve(cbam_y_true, cbam_y_pred_probs)
roc_auc_cbam = roc_auc_score(cbam_y_true, cbam_y_pred_probs)
plt.plot(fpr_cbam, tpr_cbam, label=f'CBAM (AUC={roc_auc_cbam:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid()
plt.savefig('/content/drive/My Drive/saved_models/combined_roc.png')
plt.show()

# PR Curve
plt.figure(figsize=(10, 6))
precision_danet, recall_danet, _ = precision_recall_curve(danet_y_true, danet_y_pred_probs)
pr_auc_danet = np.trapz(precision_danet, recall_danet)
plt.plot(recall_danet, precision_danet, label=f'DANet (AUC={pr_auc_danet:.2f})')

precision_panet, recall_panet, _ = precision_recall_curve(panet_y_true, panet_y_pred_probs)
pr_auc_panet = np.trapz(precision_panet, recall_panet)
plt.plot(recall_panet, precision_panet, label=f'PANet (AUC={pr_auc_panet:.2f})')

precision_cbam, recall_cbam, _ = precision_recall_curve(cbam_y_true, cbam_y_pred_probs)
pr_auc_cbam = np.trapz(precision_cbam, recall_cbam)
plt.plot(recall_cbam, precision_cbam, label=f'CBAM (AUC={pr_auc_cbam:.2f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR Curve Comparison')
plt.legend()
plt.grid()
plt.savefig('/content/drive/My Drive/saved_models/combined_pr.png')
plt.show()